Instalação do Qdrant Client

In [1]:
%pip install qdrant-client fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 14.5 MB/s eta 0:00:00


Conectando ao Qdrant Cloud

In [2]:
from qdrant_client import QdrantClient
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()  # carrega o .env, para carregar a URL e a API key sem vazar elas

# connect to Qdrant Cloud
client = QdrantClient(
    url=os.getenv("QDRANT_URL"), # Adicionando o Endpoint Cluster, que foi obtido na etapa anterior
    api_key=os.getenv("QDRANT_API_KEY"), # Adicionando a API key, que foi obtida na etapa anterior
)

Criando a coleção

In [3]:
from qdrant_client.models import Distance, VectorParams

# create collection
client.create_collection( # envia uma requisição para o servidor criar uma nova coleção
    collection_name="foods", # Define o nome da coleção
    vectors_config=VectorParams(size=384, distance=Distance.COSINE), # os vetores da coleção terão 384 dimensões e a similaridade entre eles é calculada com cosseno
)

True

Populando a coleção

In [23]:
from qdrant_client.models import PointStruct
from fastembed import TextEmbedding

# load the embedding model
model = TextEmbedding('BAAI/bge-small-en-v1.5') # esse modelo 'BAAI/bge-small-en-v1.5' converte textos em vetores de 384 dimensões

food_items = [ # Lista de comidas seguindo o formato: ("Nome alimento (medidad do alimento), g de carboidrato, g de proteina, g de goruda")
    ("Arroz cozido (100g)", 28, 2.7, 0.3),
    ("Banana (1 média)", 27, 1.3, 0.4),
    ("Aveia (40g)", 24, 5.0, 3.0),
    ("Pão francês (1 un)", 28, 5.0, 1.5),
    ("Batata doce (100g)", 20, 1.6, 0.1),
    ("Macarrão cozido (100g)", 31, 5.8, 1.1),
    ("Arroz integral cozido (100g)", 26, 2.6, 1.0),
    ("Quinoa cozida (100g)", 21, 4.4, 1.9),
    ("Cuscuz de milho (100g)", 25, 2.2, 0.6),
    ("Granola (30g)", 20, 3.0, 4.5),
    ("Maçã (1 média)", 25, 0.5, 0.3),
    ("Laranja (1 média)", 15, 1.2, 0.2),
    ("Mamão (100g)", 11, 0.5, 0.1),
    ("Manga (100g)", 15, 0.8, 0.4),
    ("Abacaxi (100g)", 13, 0.5, 0.1),
    ("Morango (100g)", 8, 0.7, 0.3),
    ("Uva (100g)", 18, 0.6, 0.2),
    ("Abacate (100g)", 9, 2.0, 15.0),
    ("Feijão carioca cozido (100g)", 14, 4.8, 0.5),
    ("Lentilha cozida (100g)", 20, 9.0, 0.4),
    ("Grão-de-bico cozido (100g)", 27, 8.9, 2.6),
    ("Brócolis cozido (100g)", 7, 2.8, 0.4),
    ("Cenoura cozida (100g)", 10, 0.9, 0.2),
    ("Abobrinha cozida (100g)", 4, 1.2, 0.3),
    ("Tomate (100g)", 5, 0.9, 0.2),
    ("Peito de frango grelhado (100g)", 0, 31.0, 3.6),
    ("Carne bovina magra grelhada (100g)", 0, 26.0, 10.0),
    ("Carne suína magra (100g)", 0, 27.0, 7.0),
    ("Peixe tilápia grelhado (100g)", 0, 26.0, 2.7),
    ("Salmão grelhado (100g)", 0, 25.0, 13.0),
    ("Ovo inteiro (1 un)", 1, 6.0, 5.0),
    ("Clara de ovo (1 un)", 0, 3.6, 0.0),
    ("Leite integral (200ml)", 10, 6.0, 6.0),
    ("Leite desnatado (200ml)", 10, 6.8, 0.4),
    ("Iogurte natural integral (170g)", 8, 6.0, 5.0),
    ("Iogurte natural desnatado (170g)", 9, 8.0, 0.5),
    ("Queijo muçarela (30g)", 1, 7.0, 6.0),
    ("Queijo cottage (100g)", 3, 11.0, 4.0),
    ("Azeite de oliva (10g)", 0, 0.0, 10.0),
    ("Manteiga (10g)", 0, 0.1, 8.2),
    ("Amendoim (30g)", 6, 7.0, 14.0),
    ("Castanha de caju (30g)", 9, 5.0, 13.0),
    ("Nozes (30g)", 4, 4.0, 20.0),
    ("Semente de chia (15g)", 5, 3.0, 4.5),
    ("Pasta de amendoim (30g)", 7, 8.0, 15.0),
    ("Chocolate ao leite (25g)", 14, 2.0, 8.0),
    ("Biscoito recheado (30g)", 22, 2.0, 6.0),
    ("Refrigerante (350ml)", 37, 0.0, 0.0),
]


def food_to_text(item): # Função para transformar os números dos macronutrientes em texto, para que o embedding consiga entender
    name, carbs, protein, fat = item
    return f"Alimento: {name}. Carboidratos: {carbs} g. Proteínas: {protein} g. Gorduras: {fat} g."

# embedding generator
points = []
texts = [food_to_text(item) for item in food_items] # Transforma cada item da lista em texto
embeddings = model.embed(texts) # Gera o embedding para cada item da lista de comidas

for i, embedding in enumerate(embeddings):
    vector = embedding.tolist() # Converte o embedding para uma lista Python, formato exigido pelo Qdrant
    name, carbs, protein, fat = food_items[i]
    point = PointStruct( # Cria um ponto que será armazenado no Qdrant
        id=i,
        vector=vector,
        payload={ # Define o payload, que são os metadados associados ao vetor
            "name": name,
            "carbs_g": carbs,
            "protein_g": protein,
            "fat_g": fat,
        },
    )
    points.append(point) # Adiciona o ponto criado à lista de pontos

# upsert points to collection
client.upsert( # Chama o método upsert do cliente Qdrant
    collection_name="foods", # Especifica a coleção em que os pontos serão armazenados
    points=points, # Envia a lista de ponto
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

Pesquisando o menu de itens

In [26]:
# generate query embedding
query_text = "Quero um alimento com Carboidratos: 25 g. Proteínas: 7 g. Gorduras: 3 g." # Texto da consulta que será usado para encontrar os itens semelhantes
query_vector = next(iter(model.embed(query_text))) # Gera o embedding do texto de consulta

# search for similar menu items
results = client.query_points( # Chama o método de busca por similaridade vetorial no Qdrant
    collection_name="foods", # Define a coleção onde a busca será realizada
    query=query_vector, # Define o vector de consulta que será comparado com os vetores armazenados
    with_payload=True,  # Define que os payloads dos metadados devem ser retornados
    limit=20 # Explicarei mais adiante porque o limite foi colocado em 20 e não em 5, como no exemplo anterior
)

# print results
for r in results.points:
    p = r.payload
    print(f"Alimento: {p.get('name')}")
    print(f"Score: {r.score}") # Imprime o score de similaridade entre o vetor de consulta e o vetor do item
    print(f"Macros: {p.get('carbs_g')}C / {p.get('protein_g')}P / {p.get('fat_g')}G (g)")
    print("-"*50)

Alimento: Ovo inteiro (1 un)
Score: 0.9481408
Macros: 1C / 6.0P / 5.0G (g)
--------------------------------------------------
Alimento: Pão francês (1 un)
Score: 0.94290215
Macros: 28C / 5.0P / 1.5G (g)
--------------------------------------------------
Alimento: Granola (30g)
Score: 0.9372616
Macros: 20C / 3.0P / 4.5G (g)
--------------------------------------------------
Alimento: Refrigerante (350ml)
Score: 0.9306164
Macros: 37C / 0.0P / 0.0G (g)
--------------------------------------------------
Alimento: Nozes (30g)
Score: 0.92445475
Macros: 4C / 4.0P / 20.0G (g)
--------------------------------------------------
Alimento: Manteiga (10g)
Score: 0.92428464
Macros: 0C / 0.1P / 8.2G (g)
--------------------------------------------------
Alimento: Aveia (40g)
Score: 0.92370355
Macros: 24C / 5.0P / 3.0G (g)
--------------------------------------------------
Alimento: Queijo muçarela (30g)
Score: 0.92339593
Macros: 1C / 7.0P / 6.0G (g)
--------------------------------------------------


Achei o resultado obtido insatisfatório, por mais que achei que pudesse ocorrer isso, por estar tratando com um exemplo muito mais numérico do que semântico igual foi o anterior. Então uma forma que achei foi fazer um "reranked", isto é, pegar os itens devolvidos pelo modelo como mais semelhantes a pesquisa e ranquear eles novamente com base nos macronutrientes alvos que foram passados na pesquisa, por isso coloquei o limite de 20 anteriormente, para que tivesse mais opções para serem ranqueadas e daí sim fazer um top5 de alimentos mais semelhantes

In [27]:
import math

target = {"carbs_g": 25, "protein_g": 7, "fat_g": 5} # macronutrientes da pesquisa

def macro_dist(payload):
    return math.sqrt( # Tratei payload e target como vetores e fiz a distância entre os componentes (carbs_g, protein_g, fat_g) entre eles
        (payload["carbs_g"] - target["carbs_g"])**2 +
        (payload["protein_g"] - target["protein_g"])**2 +
        (payload["fat_g"] - target["fat_g"])**2
    )

reranked = sorted(results.points, key=lambda r: macro_dist(r.payload))[:5] # Ranqueando novamente os alimentos

for r in reranked:
    p = r.payload
    print(f"Alimento: {p.get('name','N/A')}")
    print(f"Score (semântico): {r.score}")
    print(f"Distância macros: {macro_dist(p):.2f}")
    print(f"Macros: {p.get('carbs_g')}C / {p.get('protein_g')}P / {p.get('fat_g')}G (g)")
    print("-" * 50)


Alimento: Aveia (40g)
Score (semântico): 0.92370355
Distância macros: 3.00
Macros: 24C / 5.0P / 3.0G (g)
--------------------------------------------------
Alimento: Grão-de-bico cozido (100g)
Score (semântico): 0.9157156
Distância macros: 3.66
Macros: 27C / 8.9P / 2.6G (g)
--------------------------------------------------
Alimento: Pão francês (1 un)
Score (semântico): 0.94290215
Distância macros: 5.02
Macros: 28C / 5.0P / 1.5G (g)
--------------------------------------------------
Alimento: Biscoito recheado (30g)
Score (semântico): 0.913288
Distância macros: 5.92
Macros: 22C / 2.0P / 6.0G (g)
--------------------------------------------------
Alimento: Granola (30g)
Score (semântico): 0.9372616
Distância macros: 6.42
Macros: 20C / 3.0P / 4.5G (g)
--------------------------------------------------
